In [1]:
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
from factor_analyzer.factor_analyzer import calculate_kmo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# 0. CARGAR DATASET
# =======================================
# df = pd.read_csv("../data_collection_peru_csv/data_merge/COLLECT_COMP_PERU.csv")
df = pd.read_csv("../data_ready/COLLECT_COMP_dataset_treated.csv")

In [3]:
# 0. IDENTIFICAR NUMERO DE VALORES EN CADA VARIABLE CATEGORICA
# =======================================

# Ver el número de categorías únicas en cada columna de tipo 'object'
categorical_cols = df.select_dtypes(include=['object', 'category'])
for col in categorical_cols:
    print(f'{col}: {df[col].nunique()} categorías únicas')


STRATUM: 4 categorías únicas
BOOKID: 66 categorías únicas
RDESIGN: 3 categorías únicas
RCORE_TEST: 7 categorías únicas
RCORE_PERF: 4 categorías únicas
RS1_LEV: 3 categorías únicas
RS1_TEST: 7 categorías únicas
RCO1S_PERF: 4 categorías únicas
RS2_LEV: 3 categorías únicas
RS2_TEST: 7 categorías únicas
MPATH: 3 categorías únicas
MCORE_ItemSet: 4 categorías únicas
MCORE_TEST: 17 categorías únicas
MS1_ItemSet: 4 categorías únicas
MS1_TEST: 17 categorías únicas
MS1_LEV: 4 categorías únicas
MS2_ItemSet: 4 categorías únicas
MS2_TEST: 17 categorías únicas
MS2_LEV: 5 categorías únicas
CM033Q01S: 3 categorías únicas
DM033Q01R: 5 categorías únicas
CM474Q01S: 3 categorías únicas
DM474Q01R: 9 categorías únicas
DM155Q02C: 6 categorías únicas
CM155Q01S: 3 categorías únicas
CM155Q04S: 3 categorías únicas
DM155Q04RA: 3 categorías únicas
DM155Q04RB: 3 categorías únicas
DM155Q04RC: 3 categorías únicas
DM155Q04RD: 3 categorías únicas
CM411Q01S: 3 categorías únicas
CM411Q02S: 3 categorías únicas
DM411Q02R: 

In [4]:
# 0. CARGAR DATASET
# =======================================
from collections import defaultdict

groups = defaultdict(list)

for col in categorical_cols:
    groups[frozenset(df[col].dropna().unique())].append(col)

for categories, cols in groups.items():
    if len(cols) > 1:
        print("Columnas con las mismas categorías:", cols)


Columnas con las mismas categorías: ['RCORE_PERF', 'RCO1S_PERF']
Columnas con las mismas categorías: ['RS1_LEV', 'RS2_LEV']
Columnas con las mismas categorías: ['MCORE_ItemSet', 'MS1_ItemSet', 'MS2_ItemSet']
Columnas con las mismas categorías: ['MCORE_TEST', 'MS1_TEST', 'MS2_TEST']
Columnas con las mismas categorías: ['CM033Q01S', 'CM474Q01S', 'CM155Q01S', 'CM155Q04S', 'CM411Q01S', 'CM411Q02S', 'CM803Q01S', 'CM442Q02S', 'CM034Q01S', 'CM305Q01S', 'CM496Q01S', 'CM496Q02S', 'CM423Q01S', 'CM192Q01S', 'CM603Q01S', 'CM571Q01S', 'CM564Q01S', 'CM564Q02S', 'CM447Q01S', 'CM273Q01S', 'CM408Q01S', 'CM420Q01S', 'CM446Q01S', 'DM446Q02C', 'CM559Q01S', 'DM828Q02C', 'CM828Q03S', 'CM464Q01S', 'CM800Q01S', 'CM982Q01S', 'CM982Q02S', 'CM982Q03S', 'CM982Q04S', 'CM992Q01S', 'CM992Q02S', 'DM992Q03C', 'CM915Q01S', 'CM915Q02S', 'CM906Q01S', 'DM00KQ02C', 'CM909Q01S', 'CM909Q02S', 'CM909Q03S', 'CM949Q01S', 'CM949Q02S', 'CM00GQ01S', 'DM955Q01C', 'DM955Q02C', 'DM998Q02C', 'CM905Q01S', 'DM905Q02C', 'CM919Q01S', 'CM9

In [5]:
# ----------------------------
# VERIFICAR ESTRUCTURAS
# ----------------------------
df['DM155Q02C'].unique()

array(['Missing', '00 - No credit', '12 - Partial credit',
       '11 - Partial credit', '21 - Full credit', '13 - Partial credit'],
      dtype=object)

In [6]:
# ----------------------------
# LISTA DE PATRONES LIKERT
# ----------------------------
likert_patterns = [
    ['Strongly agree', 'Agree', 'Disagree', 'Strongly disagree'],
    ['Strongly Agree', 'Agree', 'Disagree', 'Strongly Disagree'],
    ['1', '2', '3', '4'],
    ['1', '2', '3', '4', '5'],
    ['Always', 'Often', 'Sometimes', 'Never'],
    ['All of the time', 'Most of the time', 'Some of the time', 'None of the time'],
    
    # Personales
    ['Missing', 'Strongly agree', 'Agree', 'Strongly disagree', 'Disagree'], # ST267Q01JA
    ['Missing', 'Strongly agree', 'Agree', 'Strongly disagree', 'Disagree'], # ST267Q01JA
    
    # ['Missing', 'Strongly agree', 'Agree', 'Disagree', 'Neither agree nor disagree', 'Strongly disagree'], # ST307Q01JA
    # ['Missing', '0 - No credit', '1 - Full credit'], # CM033Q01S
    # ['Missing', 'B', 'D', 'A', 'C'], # DM033Q01R
    # ['Missing', '0 - No credit', '1 - Partial credit', '2 - Full credit'], # DM906Q02C
    ['Missing', 'Checked', 'Not Checked'], # DMA114Q03RA
    ['Missing', 'Yes', 'No'], # DMA103Q04RA
    
    # ['Missing', 'Strongly agree', 'Agree', 'Disagree', 'Neither agree nor disagree', 'Strongly disagree'], # ST307Q01JA
    # ['Missing', '00 - No credit', '12 - Partial credit', '11 - Partial credit', '21 - Full credit', '13 - Partial credit'], # DM155Q02C
    # ['Low Level', 'Not used', 'Medium Level', 'High Level'], # RCORE_PERF
    # ['Low', 'Not used', 'High'], # RS1_LEV
    # ['B', 'A', 'C', 'Not used'], # MCORE_ItemSet
    
]

# Convertir patrones a sets para comparación
likert_sets = [set(p) for p in likert_patterns]

# ----------------------------------------------------------
# FUNCIÓN: IDENTIFICAR COLUMNAS LIKERT Y CONTAR
# ----------------------------------------------------------
def identify_likert_columns(df):
    likert_cols = []
    likert_count_by_pattern = {i: 0 for i in range(len(likert_patterns))}

    for col in df.columns:
        if df[col].dtype == "object":
            unique_vals = set(df[col].dropna().astype(str))

            for idx, pattern_set in enumerate(likert_sets):
                if unique_vals == pattern_set:
                    likert_cols.append(col)
                    likert_count_by_pattern[idx] += 1
                    break

    total_likert = len(likert_cols)
    return likert_cols, total_likert, likert_count_by_pattern

# ----------------------------
# EJEMPLO DE USO
# ----------------------------

likert_cols, total_likert, count_by_pattern = identify_likert_columns(df)

print("Columnas Likert detectadas:", likert_cols)
print("Total de columnas Likert:", total_likert)
print("Cantidad por patrón de escala:")
for i, count in count_by_pattern.items():
    print(f"  Patrón {i+1} ({likert_patterns[i]}): {count}")


Columnas Likert detectadas: ['DMA114Q03RA', 'DMA114Q03RB', 'DMA114Q03RC', 'DMA114Q03RD', 'DMA114Q03RE', 'DMA103Q04RA', 'DMA103Q04RB', 'DMA120Q02RA', 'DMA120Q02RB', 'DMA120Q02RC', 'DMA120Q02RD', 'DMA120Q02RE', 'DMA120Q03RA', 'DMA120Q03RB', 'DMA120Q03RC', 'DMA120Q03RD', 'DMA102Q01RA', 'DMA102Q01RB', 'DMA102Q01RC', 'DMA102Q01RD', 'DMA102Q01RE', 'DMA108Q03RA', 'DMA108Q03RB', 'DMA108Q03RC', 'DMA108Q03RD', 'DMA108Q03RE', 'DMA108Q03RF', 'DMA108Q03RG', 'DMA137Q03RA', 'DMA137Q03RB', 'DMA137Q03RC', 'DMA137Q03RD', 'DMA149Q01RA', 'DMA149Q01RB', 'DMA149Q01RC', 'DMA149Q01RD', 'DR590Q04R', 'DR590Q27R', 'DR590Q03R', 'DR590Q59R', 'DR590Q33R', 'DR590Q20R', 'DR590Q24R', 'DR590Q43R', 'DR590Q52R', 'DR590Q35R', 'DR590Q61R', 'DR590Q02R', 'DR590Q18R', 'DR590Q10R', 'DR590Q37R', 'DR590Q42R', 'DR590Q45R', 'DR590Q25R', 'DR590Q13R', 'DR590Q57R', 'DR590Q41R', 'DR590Q60R', 'DR590Q05R', 'DR590Q01R', 'DR590Q22R', 'DR590Q47R', 'DR590Q34R', 'DR590Q23R', 'DR590Q38R', 'DR590Q55R', 'DR590Q14R', 'DR590Q65R', 'DR590Q26R', 'D

In [7]:
# PASO 1: SELECCIÓN DE VARIABLES PARA AFE
# ========================================
# Selecciona variables que teóricamente midan constructos psicológicos/educativos
# Ejemplo con variables de PISA sobre motivación y autoeficacia

# afe_variables = [
#     # Variables de motivación
#     'ST352Q06JA', 'ST354Q05JA', 'ST355Q05JA', 'MATHMOT',
#     # Variables de autoeficacia
#     'CREATEFF', 'SDLEFF', 'EFFORT1', 'EFFORT2', 'MATHEFF', 'MATHEF21',
#     # Variables de apoyo escolar
#     'FAMSUP', 'TEACHSUP', 'FAMSUPSL', 'ST270Q02JA', 'ST270Q03JA', 
#     # Añade más según tu dataset
# ]

afe_variables = likert_cols # Adicionado para probar con mas variables

# Filtrar solo las variables que existen en tu dataset
afe_variables = [var for var in afe_variables if var in df.columns]
df_afe = df[afe_variables].dropna()

print(f"Variables seleccionadas para AFE: {len(afe_variables)}")
print(f"Observaciones completas: {len(df_afe)}")

Variables seleccionadas para AFE: 393
Observaciones completas: 6968


In [8]:
# ========================================
# LISTA DE VALORES EN VARIABLES CATEGORICAS
# ========================================

likert_maps = [
    # 4-puntos clásicos
    {'Strongly agree':4, 'Agree':3, 'Disagree':2, 'Strongly disagree':1},
    {'Strongly Agree':4, 'Agree':3, 'Disagree':2, 'Strongly Disagree':1},

    # Números ya en strings
    {'1':1, '2':2, '3':3, '4':4},
    {'1':1, '2':2, '3':3, '4':4, '5':5},

    # Frecuencia
    {'Always':4, 'Often':3, 'Sometimes':2, 'Never':1},
    {'All of the time':4, 'Most of the time':3, 'Some of the time':2, 'None of the time':1},

    # Personales
    {'Missing': np.nan, 'Strongly agree':4, 'Agree':3, 'Disagree':2, 'Strongly disagree':1},
    {'Missing': np.nan, 'Strongly agree':4, 'Agree':3, 'Neither agree nor disagree':2, 'Disagree':1, 'Strongly disagree':0},

    {'Missing': np.nan, '0 - No credit':0, '1 - Full credit':1},

    {'Missing': np.nan, 'A':4, 'B':3, 'C':2, 'D':1},

    {'Missing': np.nan, '0 - No credit':0, '1 - Partial credit':1, '2 - Full credit':2},

    {'Missing': np.nan, 'Checked':1, 'Not Checked':0},

    {'Missing': np.nan, 'Yes':1, 'No':0},

    {'Missing': np.nan, 'Strongly agree':4, 'Agree':3, 'Neither agree nor disagree':2, 'Disagree':1, 'Strongly disagree':0},
]


In [10]:
df_afe

,DMA114Q03RA,DMA114Q03RB,DMA114Q03RC,DMA114Q03RD,DMA114Q03RE,DMA103Q04RA,DMA103Q04RB,DMA120Q02RA,DMA120Q02RB,DMA120Q02RC,...,SC221Q01JA,SC221Q02JA,SC221Q03JA,SC221Q04JA,SC155Q06HA,SC155Q07HA,SC155Q08HA,SC155Q09HA,SC155Q10HA,SC155Q11HA
0,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,...,Yes,Yes,Yes,Yes,Strongly agree,Agree,Agree,Agree,Agree,Strongly agree
1,Missing,Missing,Missing,Missing,Missing,Yes,No,Missing,Missing,Missing,...,Yes,Yes,Yes,Yes,Agree,Agree,Agree,Strongly disagree,Agree,Strongly agree
2,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,...,Yes,No,Yes,Yes,Agree,Strongly agree,Agree,Agree,Disagree,Strongly disagree
3,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,...,Yes,Yes,No,Yes,Agree,Agree,Disagree,Disagree,Strongly disagree,Strongly disagree
4,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,...,Yes,Yes,Yes,Yes,Strongly agree,Agree,Agree,Strongly agree,Agree,Strongly agree
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6963,Not Checked,Not Checked,Not Checked,Not Checked,Checked,Missing,Missing,Not Checked,Not Checked,Checked,...,Yes,Yes,No,Yes,Disagree,Disagree,Disagree,Disagree,Disagree,Agree
6964,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,...,Yes,Yes,Yes,Yes,Disagree,Disagree,Disagree,Disagree,Disagree,Strongly agree
6965,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,...,Yes,Yes,Yes,Yes,Strongly agree,Agree,Agree,Agree,Strongly agree,Strongly agree
6966,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,Missing,...,Yes,Yes,Yes,Yes,Disagree,Agree,Disagree,Strongly disagree,Strongly disagree,Agree


In [11]:
# ========================================
# LISTA DE VALORES EN VARIABLES CATEGORICAS
# ========================================

def convert_likert_column(series, likert_maps):
    # Tomamos los valores únicos excluyendo NaN
    unique_vals = set(series.dropna().unique())

    for mapping in likert_maps:
        # Verificamos si todos los valores están en el diccionario de mapeo
        if unique_vals.issubset(mapping.keys()):
            return series.map(mapping).astype(float)  # convertir a float
    
    # Si no coincide con ningún patrón, intentar convertir a numérico
    return pd.to_numeric(series, errors='coerce')

# Aplicar a todo el DataFrame
for col in df_afe.columns:
    df_afe[col] = convert_likert_column(df_afe[col], likert_maps)

# Comprobar
print(df_afe.head())
print(df_afe.dtypes)
print(df_afe.isna().sum())



   DMA114Q03RA  DMA114Q03RB  DMA114Q03RC  DMA114Q03RD  DMA114Q03RE  \
0          NaN          NaN          NaN          NaN          NaN   
1          NaN          NaN          NaN          NaN          NaN   
2          NaN          NaN          NaN          NaN          NaN   
3          NaN          NaN          NaN          NaN          NaN   
4          NaN          NaN          NaN          NaN          NaN   

   DMA103Q04RA  DMA103Q04RB  DMA120Q02RA  DMA120Q02RB  DMA120Q02RC  ...  \
0          NaN          NaN          NaN          NaN          NaN  ...   
1          1.0          0.0          NaN          NaN          NaN  ...   
2          NaN          NaN          NaN          NaN          NaN  ...   
3          NaN          NaN          NaN          NaN          NaN  ...   
4          NaN          NaN          NaN          NaN          NaN  ...   

   SC221Q01JA  SC221Q02JA  SC221Q03JA  SC221Q04JA  SC155Q06HA  SC155Q07HA  \
0         1.0         1.0         1.0         1.0  

In [12]:
df_afe = df_afe.fillna(df_afe.median())
df_afe

,DMA114Q03RA,DMA114Q03RB,DMA114Q03RC,DMA114Q03RD,DMA114Q03RE,DMA103Q04RA,DMA103Q04RB,DMA120Q02RA,DMA120Q02RB,DMA120Q02RC,...,SC221Q01JA,SC221Q02JA,SC221Q03JA,SC221Q04JA,SC155Q06HA,SC155Q07HA,SC155Q08HA,SC155Q09HA,SC155Q10HA,SC155Q11HA
0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,4.0,3.0,3.0,3.0,3.0,4.0
1,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,3.0,3.0,3.0,1.0,3.0,4.0
2,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,3.0,4.0,3.0,3.0,2.0,1.0
3,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,3.0,3.0,2.0,2.0,1.0,1.0
4,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,4.0,3.0,3.0,4.0,3.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6963,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,0.0,1.0,2.0,2.0,2.0,2.0,2.0,3.0
6964,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,4.0
6965,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,4.0,3.0,3.0,3.0,4.0,4.0
6966,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,2.0,3.0,2.0,1.0,1.0,3.0


In [12]:
# LIMIEZA DE MISSINGS DEL DATASET - PARA HACER PRUEBAS
# -------------------------------------------

# Supongamos que tu dataset se llama df
# df = pd.read_csv("pisa2022.csv")  # ejemplo de lectura

# Crear listas para variables imputadas y descartadas
vars_imputadas = []
vars_descartadas = []

# Crear copia del dataset para trabajar
df_limpio = df_afe.copy()

# Iterar sobre las columnas
for col in df_limpio.columns:
    total = len(df_limpio[col])
    missing = df_limpio[col].isna().sum()  # Contar None/NaN
    missing_pct = missing / total * 100

    if missing_pct > 20:
        # Descartar variable
        df_limpio.drop(columns=col, inplace=True)
        vars_descartadas.append(col)
    elif 10 <= missing_pct <= 20:
        # Imputar con la moda (valor más frecuente)
        moda = df_limpio[col].mode(dropna=True)[0]  # Evitar NaN
        df_limpio[col].fillna(moda, inplace=True)
        vars_imputadas.append(col)
    # Si <10%, no hacemos nada, dejamos los missing tal cual

# Mostrar resultados
print("Variables descartadas (>20% missing):")
print(vars_descartadas)
print("\nVariables imputadas (10-20% missing):")
print(vars_imputadas)

# df_limpio es tu dataset listo para análisis


Variables descartadas (>20% missing):
[]

Variables imputadas (10-20% missing):
[]


In [13]:
# df_afe = df_limpio

df_afe

,DMA114Q03RA,DMA114Q03RB,DMA114Q03RC,DMA114Q03RD,DMA114Q03RE,DMA103Q04RA,DMA103Q04RB,DMA120Q02RA,DMA120Q02RB,DMA120Q02RC,...,SC221Q01JA,SC221Q02JA,SC221Q03JA,SC221Q04JA,SC155Q06HA,SC155Q07HA,SC155Q08HA,SC155Q09HA,SC155Q10HA,SC155Q11HA
0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,4.0,3.0,3.0,3.0,3.0,4.0
1,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,3.0,3.0,3.0,1.0,3.0,4.0
2,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,3.0,4.0,3.0,3.0,2.0,1.0
3,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,3.0,3.0,2.0,2.0,1.0,1.0
4,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,4.0,3.0,3.0,4.0,3.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6963,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,0.0,1.0,2.0,2.0,2.0,2.0,2.0,3.0
6964,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,4.0
6965,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,4.0,3.0,3.0,3.0,4.0,4.0
6966,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,2.0,3.0,2.0,1.0,1.0,3.0


In [14]:
# PASO 2: VERIFICACIÓN DE SUPUESTOS
# ==================================

# 2.1 Test de esfericidad de Bartlett
# H0: La matriz de correlación es una matriz identidad
chi_square_value, p_value = calculate_bartlett_sphericity(df_afe)
print(f'\nTest de Bartlett:')
print(f'Chi-cuadrado: {chi_square_value:.2f}')
print(f'p-value: {p_value:.4f}')
if p_value < 0.05:
    print("✓ Se rechaza H0: Las variables están correlacionadas (bueno para AFE)")
else:
    print("✗ No se rechaza H0: Las variables no están suficientemente correlacionadas")

# 2.2 Kaiser-Meyer-Olkin (KMO)
# Mide la adecuación muestral para AFE
kmo_all, kmo_model = calculate_kmo(df_afe)
print(f'\nKMO General: {kmo_model:.3f}')
print("Interpretación KMO:")
print("  > 0.9: Excelente")
print("  0.8-0.9: Bueno")
print("  0.7-0.8: Aceptable")
print("  0.6-0.7: Mediocre")
print("  < 0.6: Inadecuado")

# KMO por variable
kmo_per_variable = pd.DataFrame({
    'Variable': afe_variables,
    'KMO': kmo_all
})
print("\nKMO por variable (las < 0.5 deberían eliminarse):")
print(kmo_per_variable[kmo_per_variable['KMO'] < 0.6])


Test de Bartlett:
Chi-cuadrado: 879373.25
p-value: 0.0000
✓ Se rechaza H0: Las variables están correlacionadas (bueno para AFE)

KMO General: 0.800
Interpretación KMO:
  > 0.9: Excelente
  0.8-0.9: Bueno
  0.7-0.8: Aceptable
  0.6-0.7: Mediocre
  < 0.6: Inadecuado

KMO por variable (las < 0.5 deberían eliminarse):
        Variable       KMO
5    DMA103Q04RA  0.523576
6    DMA103Q04RB  0.500124
16   DMA102Q01RA  0.557666
19   DMA102Q01RD  0.521567
20   DMA102Q01RE  0.530609
33   DMA149Q01RB  0.581275
116   DR446Q03RC  0.557804
128   DR432Q06RC  0.531972
130   DR432Q06RE  0.509519
131   DR432Q06RF  0.576463
132   DR432Q06RG  0.478924
133   DR432Q06RH  0.588335
142   DS643Q02RA  0.548192
143   DS643Q02RB  0.591039
144   DS643Q02RC  0.592050
145   DS643Q02RD  0.550328
322   SC035Q01NB  0.478565
324   SC035Q02TB  0.452647
326   SC035Q03TB  0.598073
328   SC035Q04TB  0.558543
338   SC035Q09NB  0.571033
344   SC181Q02JA  0.490670
345   SC181Q03JA  0.417155
368   SC215Q04JA  0.589475
369   SC

c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(


In [15]:
# PASO 3: DETERMINAR NÚMERO DE FACTORES
# =====================================

# 3.1 Análisis paralelo (más robusto)
from factor_analyzer import FactorAnalyzer
# from factor_analyzer.utils import random_correlation_matrix

def parallel_analysis(df, n_iterations=100):
    """Análisis paralelo para determinar número de factores"""
    n_obs, n_vars = df.shape
    
    # Calcular autovalores reales
    fa_real = FactorAnalyzer(n_factors=n_vars, rotation=None)
    fa_real.fit(df)
    ev_real = fa_real.get_eigenvalues()[0]
    
    # Generar autovalores aleatorios
    ev_simulated = np.zeros((n_iterations, n_vars))
    for i in range(n_iterations):
        random_data = np.random.normal(size=(n_obs, n_vars))
        fa_sim = FactorAnalyzer(n_factors=n_vars, rotation=None)
        fa_sim.fit(pd.DataFrame(random_data))
        ev_simulated[i, :] = fa_sim.get_eigenvalues()[0]
    
    # Percentil 95 de autovalores simulados
    ev_95 = np.percentile(ev_simulated, 95, axis=0)
    
    # Determinar número de factores
    n_factors = np.sum(ev_real > ev_95)
    
    # Visualización
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(ev_real)+1), ev_real, 'bo-', label='Datos reales')
    plt.plot(range(1, len(ev_95)+1), ev_95, 'r--', label='Percentil 95 (aleatorio)')
    plt.axhline(y=1, color='gray', linestyle=':', label='Kaiser (autovalor=1)')
    plt.xlabel('Número de factor')
    plt.ylabel('Autovalor')
    plt.title('Análisis Paralelo')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return n_factors

n_factors_parallel = parallel_analysis(df_afe)
print(f"\nNúmero de factores sugerido por análisis paralelo: {n_factors_parallel}")

# 3.2 Scree plot
fa = FactorAnalyzer(n_factors=len(afe_variables), rotation=None)
fa.fit(df_afe)
ev, v = fa.get_eigenvalues()

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(ev)+1), ev, 'bo-')
plt.axhline(y=1, color='r', linestyle='--', label='Criterio de Kaiser')
plt.xlabel('Número de factor')
plt.ylabel('Autovalor')
plt.title('Scree Plot')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\d

KeyboardInterrupt: 

In [14]:
# PASO 4: EXTRACCIÓN DE FACTORES
# ===============================

# Usar el número de factores determinado
n_factors = n_factors_parallel  # o ajusta manualmente basado en teoría

# Probar diferentes métodos de rotación
rotations = ['varimax', 'promax', 'oblimin']
results = {}

for rotation in rotations:
    fa = FactorAnalyzer(n_factors=n_factors, rotation=rotation, method='principal')
    fa.fit(df_afe)
    
    # Guardar resultados
    results[rotation] = {
        'loadings': fa.loadings_,
        'communalities': fa.get_communalities(),
        'uniqueness': fa.get_uniquenesses(),
        'variance': fa.get_factor_variance()
    }
    
    print(f"\n{'='*50}")
    print(f"ROTACIÓN: {rotation.upper()}")
    print(f"{'='*50}")
    
    # Matriz de cargas factoriales
    loadings_df = pd.DataFrame(
        fa.loadings_,
        index=afe_variables,
        columns=[f'Factor{i+1}' for i in range(n_factors)]
    )
    
    # Mostrar solo cargas > 0.3
    def highlight_loadings(val):
        if abs(val) > 0.5:
            return 'font-weight: bold; color: red'
        elif abs(val) > 0.3:
            return 'font-weight: bold; color: blue'
        return ''
    
    print("\nCargas factoriales (>0.3 en azul, >0.5 en rojo):")
    display(loadings_df.style.applymap(highlight_loadings))
    
    # Varianza explicada
    variance = fa.get_factor_variance()
    variance_df = pd.DataFrame(variance, 
                              index=['Varianza', 'Proporción', 'Acumulada'],
                              columns=[f'Factor{i+1}' for i in range(n_factors)])
    print("\nVarianza explicada:")
    print(variance_df)

c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



ROTACIÓN: VARIMAX

Cargas factoriales (>0.3 en azul, >0.5 en rojo):


C:\Users\max_a\AppData\Local\Temp\ipykernel_15452\3653750460.py:43: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(loadings_df.style.applymap(highlight_loadings))


,Factor1,Factor2,Factor3,Factor4,Factor5,Factor6,Factor7,Factor8,Factor9,Factor10,Factor11,Factor12,Factor13,Factor14,Factor15,Factor16,Factor17,Factor18,Factor19,Factor20,Factor21,Factor22,Factor23,Factor24,Factor25,Factor26
ST267Q01JA,0.040037,0.007507,-0.027603,0.035025,-0.039844,-0.042509,0.044237,-0.022578,0.038710,0.020497,-0.010366,-0.020392,0.642188,0.005755,-0.033797,0.056266,0.035743,0.015675,0.005216,0.079433,-0.030607,0.034264,-0.017971,0.009239,0.169940,-0.001705
ST267Q02JA,0.045938,0.020529,0.006768,0.026861,-0.016842,0.023349,-0.012465,0.031944,-0.031060,0.084124,0.000937,0.053926,0.557049,0.001832,-0.005748,-0.015301,0.042613,0.038023,0.049430,-0.069023,0.067946,0.109083,0.002141,-0.026313,-0.070513,0.041552
ST267Q03JA,0.067688,0.018361,-0.046517,0.105184,0.038229,0.008291,0.008544,-0.007678,0.014033,0.102412,0.028206,-0.026140,0.503997,0.021127,0.045594,-0.059032,0.047428,0.064848,0.071910,-0.024475,-0.014563,0.076557,-0.012853,-0.060918,-0.087483,-0.038690
ST267Q04JA,-0.038372,-0.010057,0.004959,-0.000083,-0.019157,0.080086,-0.019919,-0.006519,-0.002833,-0.045574,0.005477,-0.015535,-0.064203,-0.057553,0.075256,-0.007479,-0.059598,0.002269,0.015208,-0.055256,-0.018274,-0.003909,0.036478,0.044665,-0.625439,-0.037669
ST267Q05JA,0.024773,0.008994,-0.020472,0.034892,0.010803,0.012182,0.027966,-0.001480,0.000987,0.069672,-0.021636,0.052317,0.625274,0.004789,0.008976,-0.046109,0.051416,0.052350,0.038052,-0.020661,0.075714,0.091095,-0.011429,0.015847,0.003605,0.004635
ST267Q06JA,0.045878,0.020405,-0.006774,0.055385,0.019208,-0.034502,0.002407,-0.010333,-0.000142,0.025022,0.015474,0.021549,0.651426,0.032389,0.001194,0.011091,0.015211,0.009440,0.035140,0.049657,0.000311,0.078195,0.015767,-0.006828,0.121624,-0.043208
ST267Q07JA,0.004190,0.012132,0.019204,0.001670,0.027680,-0.014958,0.051644,-0.003314,0.005807,0.034273,0.005932,0.056583,0.621171,-0.020230,-0.017048,0.051217,0.003319,0.049681,0.015799,0.032011,0.015014,0.128159,0.010644,0.000317,0.117327,0.001738
ST267Q08JA,-0.002993,-0.009911,0.028058,0.010019,-0.052145,0.079071,-0.049718,0.000161,-0.030422,-0.003025,0.036684,-0.002105,-0.203024,-0.030285,-0.019459,-0.059088,-0.011974,0.007263,-0.017741,-0.014015,0.035280,-0.049275,-0.040316,0.127395,-0.646629,-0.040205
ST034Q01TA,-0.032065,-0.000311,0.065894,-0.050380,0.004495,0.033949,-0.058370,-0.017050,-0.032858,-0.697955,0.046234,-0.000854,-0.161149,-0.024794,0.038496,-0.124160,-0.043687,0.022573,0.030925,-0.078378,-0.001675,-0.045995,0.001505,0.085438,0.056574,0.006694
ST034Q02TA,0.053872,-0.001734,-0.033168,0.043661,-0.039778,0.066289,-0.117547,0.005330,-0.015458,0.580485,0.037011,0.054112,-0.019018,-0.011923,0.057142,-0.270022,0.099809,0.065639,0.119948,-0.085571,-0.019168,0.033419,-0.064260,0.055847,0.159633,-0.004074


c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



Varianza explicada:
             Factor1   Factor2   Factor3   Factor4   Factor5   Factor6  \
Varianza    4.019101  6.113044  3.487207  3.419604  3.095279  2.519800   
Proporción  0.030680  0.046664  0.026620  0.026104  0.023628  0.019235   
Acumulada   0.030680  0.077345  0.103965  0.130068  0.153696  0.172932   

             Factor7   Factor8   Factor9  Factor10  ...  Factor17  Factor18  \
Varianza    2.745081  3.018285  3.501002  2.351135  ...  2.626790  1.785712   
Proporción  0.020955  0.023040  0.026725  0.017948  ...  0.020052  0.013631   
Acumulada   0.193886  0.216927  0.243652  0.261600  ...  0.386724  0.400355   

            Factor19  Factor20  Factor21  Factor22  Factor23  Factor24  \
Varianza    2.592058  1.765783  1.450696  3.723541  1.616268  1.522205   
Proporción  0.019787  0.013479  0.011074  0.028424  0.012338  0.011620   
Acumulada   0.420142  0.433621  0.444695  0.473119  0.485457  0.497077   

            Factor25  Factor26  
Varianza    1.331102  1.660956  
Pr

... (4 outputs left out)


ROTACIÓN: OBLIMIN

Cargas factoriales (>0.3 en azul, >0.5 en rojo):


C:\Users\max_a\AppData\Local\Temp\ipykernel_15452\3653750460.py:43: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(loadings_df.style.applymap(highlight_loadings))


,Factor1,Factor2,Factor3,Factor4,Factor5,Factor6,Factor7,Factor8,Factor9,Factor10,Factor11,Factor12,Factor13,Factor14,Factor15,Factor16,Factor17,Factor18,Factor19,Factor20,Factor21,Factor22,Factor23,Factor24,Factor25,Factor26
ST267Q01JA,-0.029471,-0.004382,-0.020116,-0.005616,-0.052515,-0.014455,0.019293,-0.019608,0.026671,-0.011096,0.004370,-0.030476,0.644393,-0.005592,-0.031182,0.029142,0.018197,-0.005631,-0.002433,0.047247,-0.035724,-0.054269,-0.017894,0.042372,-0.148539,0.005833
ST267Q02JA,0.068185,0.021751,0.016112,0.025436,-0.043240,0.044389,-0.046111,0.036526,-0.031302,-0.086041,-0.001897,0.034660,0.546091,-0.008036,0.006739,0.032940,0.008394,0.016515,0.045191,-0.097671,0.061219,-0.003770,0.000551,-0.015251,0.102152,0.033384
ST267Q03JA,0.028315,0.007136,-0.046475,0.102845,0.015217,0.031271,-0.023807,-0.009614,0.011475,-0.085740,0.038531,-0.047418,0.494524,0.013833,0.044475,0.020963,0.005484,0.046044,0.066031,-0.053711,-0.030059,0.045552,-0.017766,-0.053645,0.127429,-0.047462
ST267Q04JA,0.051194,0.013900,-0.010249,0.058161,-0.027498,0.077167,-0.024656,-0.003834,0.016661,-0.060648,0.012878,-0.000788,-0.042970,-0.060787,0.095373,-0.010360,-0.066344,0.016083,0.040054,-0.071665,-0.018155,-0.110081,0.024611,0.018188,0.630476,-0.063185
ST267Q05JA,0.031926,0.003221,-0.009675,0.017263,-0.011453,0.035772,-0.007806,0.002302,0.001663,-0.048835,-0.026516,0.029160,0.621034,-0.003164,0.010321,-0.007755,0.017685,0.026251,0.020958,-0.042384,0.065530,0.033010,-0.012434,0.025710,0.040932,0.000915
ST267Q06JA,0.022279,0.007940,0.002112,0.029216,0.005515,-0.008029,-0.036634,-0.010661,-0.013980,-0.007793,0.025467,0.004594,0.650959,0.027443,0.002508,0.011553,-0.015897,-0.015339,0.030598,0.024686,-0.011289,-0.014782,0.013476,0.013985,-0.090607,-0.044883
ST267Q07JA,0.077339,0.013035,0.030604,-0.029824,0.020455,0.009360,0.017335,0.005009,0.002232,-0.034658,0.009763,0.040308,0.610525,-0.034355,-0.012713,-0.031893,-0.030619,0.026700,0.000337,0.006613,0.005165,-0.054261,0.001410,0.026487,-0.095792,-0.000195
ST267Q08JA,0.004448,0.009431,0.032971,0.056063,-0.060403,0.063855,-0.038339,-0.002517,-0.014645,-0.083814,0.036829,0.014235,-0.178702,-0.022603,-0.008664,0.048430,-0.005919,0.020159,0.001518,-0.012692,0.042645,-0.038931,-0.039536,0.093125,0.661975,-0.063853
ST034Q01TA,-0.007650,0.009769,0.026609,-0.007000,-0.013646,0.013559,-0.045171,-0.018218,-0.012943,0.732401,0.001741,-0.007672,-0.073371,-0.007931,0.010132,-0.016150,-0.008304,0.036125,0.028547,-0.039982,-0.000730,-0.034546,0.000349,0.027390,-0.031906,0.001862
ST034Q02TA,-0.009637,0.001610,0.007577,0.003625,-0.053146,0.048654,-0.125942,0.002257,-0.008324,-0.403112,0.022170,0.033891,-0.055845,-0.013787,0.042573,0.028143,0.074296,0.043175,0.117459,-0.060461,-0.026483,0.499268,-0.037576,0.035206,-0.100695,0.008021



Varianza explicada:
             Factor1   Factor2   Factor3   Factor4   Factor5   Factor6  \
Varianza    3.341030  5.638023  3.203685  3.018221  3.178665  2.506982   
Proporción  0.025504  0.043038  0.024456  0.023040  0.024265  0.019137   
Acumulada   0.025504  0.068542  0.092998  0.116038  0.140302  0.159440   

             Factor7   Factor8   Factor9  Factor10  ...  Factor17  Factor18  \
Varianza    2.798011  3.050441  3.442972  2.158385  ...  2.405260  1.768112   
Proporción  0.021359  0.023286  0.026282  0.016476  ...  0.018361  0.013497   
Acumulada   0.180799  0.204084  0.230367  0.246843  ...  0.377590  0.391087   

            Factor19  Factor20  Factor21  Factor22  Factor23  Factor24  \
Varianza    2.382146  1.791261  1.452724  1.419566  1.605730  1.472743   
Proporción  0.018184  0.013674  0.011089  0.010836  0.012257  0.011242   
Acumulada   0.409272  0.422946  0.434035  0.444871  0.457129  0.468371   

            Factor25  Factor26  
Varianza    1.368958  1.617695  
Pr

In [15]:
# PASO 5: INTERPRETACIÓN Y ETIQUETADO DE FACTORES
# ================================================

# Usar la mejor rotación (generalmente varimax para factores independientes)
best_rotation = 'varimax'
fa_final = FactorAnalyzer(n_factors=n_factors, rotation=best_rotation, method='principal')
fa_final.fit(df_afe)

# Crear dataframe de cargas para interpretación
loadings_final = pd.DataFrame(
    fa_final.loadings_,
    index=afe_variables,
    columns=[f'Factor{i+1}' for i in range(n_factors)]
)

# Identificar items dominantes por factor (carga > 0.4)
for i in range(n_factors):
    print(f"\n{'='*50}")
    print(f"FACTOR {i+1}")
    print(f"{'='*50}")
    factor_items = loadings_final[f'Factor{i+1}'].abs().sort_values(ascending=False)
    dominant_items = factor_items[factor_items > 0.4]
    print("Items con carga > 0.4:")
    for item, loading in dominant_items.items():
        print(f"  {item}: {loadings_final.loc[item, f'Factor{i+1}']:.3f}")
    
    # Aquí deberías interpretar basado en el contenido de las preguntas
    print("\nInterpretación sugerida: [Analizar contenido de items]")

c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



FACTOR 1
Items con carga > 0.4:
  ST340Q06JA: 0.594
  ST340Q05JA: 0.572
  ST340Q04JA: 0.563
  ST340Q01JA: 0.547
  ST340Q09JA: 0.519
  ST340Q07JA: 0.472
  ST340Q02JA: 0.463
  ST340Q08JA: 0.451
  ST340Q10JA: 0.438
  ST339Q06JA: 0.416
  ST342Q04JA: 0.402

Interpretación sugerida: [Analizar contenido de items]

FACTOR 2
Items con carga > 0.4:
  SC208Q08JA: 0.831
  SC208Q09JA: 0.825
  SC208Q04JA: 0.816
  SC208Q03JA: 0.791
  SC208Q07JA: 0.764
  SC208Q01JA: 0.756
  SC208Q06JA: 0.747
  SC208Q02JA: 0.746
  SC208Q05JA: 0.741

Interpretación sugerida: [Analizar contenido de items]

FACTOR 3
Items con carga > 0.4:
  ST292Q04JA: 0.690
  ST292Q06JA: 0.686
  ST292Q02JA: 0.677
  ST292Q01JA: 0.676
  ST292Q03JA: 0.665
  ST292Q05JA: 0.652
  ST268Q04JA: -0.506
  ST268Q01JA: -0.502

Interpretación sugerida: [Analizar contenido de items]

FACTOR 4
Items con carga > 0.4:
  FL169Q08JA: 0.752
  FL169Q10JA: 0.732
  FL169Q04HA: 0.683
  FL169Q05JA: 0.600
  FL169Q01HA: 0.567
  FL169Q11JA: 0.563
  FL169Q02HA: 0.41

In [19]:
# PASO 6: CÁLCULO DE PUNTUACIONES FACTORIALES
# ===========================================

# Obtener puntuaciones factoriales para cada estudiante
factor_scores = fa_final.transform(df_afe)
factor_scores_df = pd.DataFrame(
    factor_scores,
    columns=[f'Factor_{i+1}_Score' for i in range(n_factors)],
    index=df_afe.index
)

# Añadir al dataset principal
df_with_factors = pd.concat([df, factor_scores_df], axis=1)

print(f"\nPuntuaciones factoriales calculadas y añadidas al dataset")
print(f"Nuevas columnas: {factor_scores_df.columns.tolist()}")


Puntuaciones factoriales calculadas y añadidas al dataset
Nuevas columnas: ['Factor_1_Score', 'Factor_2_Score', 'Factor_3_Score', 'Factor_4_Score', 'Factor_5_Score', 'Factor_6_Score', 'Factor_7_Score', 'Factor_8_Score', 'Factor_9_Score', 'Factor_10_Score', 'Factor_11_Score', 'Factor_12_Score', 'Factor_13_Score', 'Factor_14_Score', 'Factor_15_Score', 'Factor_16_Score', 'Factor_17_Score', 'Factor_18_Score', 'Factor_19_Score', 'Factor_20_Score', 'Factor_21_Score', 'Factor_22_Score', 'Factor_23_Score', 'Factor_24_Score', 'Factor_25_Score', 'Factor_26_Score']


c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [20]:
# PASO 7: VALIDACIÓN DE LA SOLUCIÓN FACTORIAL
# ===========================================

# 7.1 Confiabilidad (Alpha de Cronbach) por factor
from scipy.stats import pearsonr

def cronbach_alpha(df):
    """Calcula el alpha de Cronbach"""
    df_corr = df.corr()
    N = df.shape[1]
    rs = np.array([])
    for i, col in enumerate(df_corr.columns):
        sum_ = df_corr[col].values.sum() - 1  # Excluir diagonal
        rs = np.append(rs, sum_)
    mean_r = np.mean(rs) / (N - 1)
    alpha = (N * mean_r) / (1 + (N - 1) * mean_r)
    return alpha

# Calcular alpha para cada factor
for i in range(n_factors):
    # Identificar items del factor
    factor_items = loadings_final[f'Factor{i+1}'].abs()
    items = factor_items[factor_items > 0.4].index.tolist()
    
    if len(items) > 1:
        alpha = cronbach_alpha(df_afe[items])
        print(f"\nFactor {i+1} - Alpha de Cronbach: {alpha:.3f}")
        print("Interpretación: ", end="")
        if alpha >= 0.9:
            print("Excelente")
        elif alpha >= 0.8:
            print("Bueno")
        elif alpha >= 0.7:
            print("Aceptable")
        else:
            print("Cuestionable - considerar revisar items")

# 7.2 Validación con split-half
from sklearn.model_selection import train_test_split

# Dividir muestra
df_train, df_test = train_test_split(df_afe, test_size=0.3, random_state=42)

# AFE en muestra de entrenamiento
fa_train = FactorAnalyzer(n_factors=n_factors, rotation=best_rotation)
fa_train.fit(df_train)

# AFE en muestra de test
fa_test = FactorAnalyzer(n_factors=n_factors, rotation=best_rotation)
fa_test.fit(df_test)

# Comparar estructuras factoriales (correlación de cargas)
for i in range(n_factors):
    corr, p = pearsonr(fa_train.loadings_[:, i], fa_test.loadings_[:, i])
    print(f"Factor {i+1} - Correlación train/test: {corr:.3f} (p={p:.4f})")


Factor 1 - Alpha de Cronbach: 0.762
Interpretación: Aceptable

Factor 2 - Alpha de Cronbach: 0.932
Interpretación: Excelente

Factor 3 - Alpha de Cronbach: 0.489
Interpretación: Cuestionable - considerar revisar items

Factor 4 - Alpha de Cronbach: 0.786
Interpretación: Aceptable

Factor 5 - Alpha de Cronbach: 0.840
Interpretación: Bueno

Factor 6 - Alpha de Cronbach: 0.775
Interpretación: Aceptable

Factor 7 - Alpha de Cronbach: 0.928
Interpretación: Excelente

Factor 8 - Alpha de Cronbach: 0.894
Interpretación: Bueno

Factor 9 - Alpha de Cronbach: 0.852
Interpretación: Bueno

Factor 10 - Alpha de Cronbach: 0.196
Interpretación: Cuestionable - considerar revisar items

Factor 11 - Alpha de Cronbach: 0.762
Interpretación: Aceptable

Factor 12 - Alpha de Cronbach: 0.555
Interpretación: Cuestionable - considerar revisar items

Factor 13 - Alpha de Cronbach: 0.689
Interpretación: Cuestionable - considerar revisar items

Factor 14 - Alpha de Cronbach: 0.861
Interpretación: Bueno

Factor 1

c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Factor 1 - Correlación train/test: 0.993 (p=0.0000)
Factor 2 - Correlación train/test: 0.960 (p=0.0000)
Factor 3 - Correlación train/test: -0.146 (p=0.0964)
Factor 4 - Correlación train/test: -0.148 (p=0.0921)
Factor 5 - Correlación train/test: -0.111 (p=0.2070)
Factor 6 - Correlación train/test: -0.107 (p=0.2226)
Factor 7 - Correlación train/test: 0.319 (p=0.0002)
Factor 8 - Correlación train/test: 0.974 (p=0.0000)
Factor 9 - Correlación train/test: 0.977 (p=0.0000)
Factor 10 - Correlación train/test: -0.038 (p=0.6658)
Factor 11 - Correlación train/test: 0.962 (p=0.0000)
Factor 12 - Correlación train/test: -0.083 (p=0.3474)
Factor 13 - Correlación train/test: 0.937 (p=0.0000)
Factor 14 - Correlación train/test: -0.038 (p=0.6632)
Factor 15 - Correlación train/test: -0.125 (p=0.1564)
Factor 16 - Correlación train/test: 0.908 (p=0.0000)
Factor 17 - Correlación train/test: -0.008 (p=0.9295)
Factor 18 - Correlación train/test: 0.945 (p=0.0000)
Factor 19 - Correlación train/test: -0.186 (p=

In [ ]:
# PASO 6.5: CREACION DE NUEVO DATASET CON VARIABLES UPDATEADAS Y REDUCIDAS
# ===========================================

def discart_var(df, discart_list):
    # Filtrar solo las columnas que realmente existen
    columnas_existentes = [col for col in discart_list if col in df.columns]
    
    # Crear nuevo DataFrame sin esas variables
    nuevo_df = df.drop(columns=columnas_existentes, errors="ignore")
    
    return nuevo_df

# discart_list = [
#     'ST270Q02JA', 'ST270Q03JA', 'TEACHSUP', 'MATHMOT', 'MATHEF21', 'MATHEFF', 'CREATEFF', 'ST355Q05JA', 'ST352Q06JA', 'ST354Q05JA', 'MATHMOT',
#     'Factor_4_Score', 'Factor_5_Score'
# ]

discart_list = [
    # Factor 1
    "ST340Q06JA", "ST340Q05JA", "ST340Q04JA", "ST340Q01JA", "ST340Q09JA", "ST340Q07JA", "ST340Q02JA", "ST340Q08JA", "ST340Q10JA", "ST339Q06JA", "ST342Q04JA",
    # Factor 2
    "SC208Q08JA", "SC208Q09JA", "SC208Q04JA", "SC208Q03JA", "SC208Q07JA", "SC208Q01JA", "SC208Q06JA", "SC208Q02JA", "SC208Q05JA",
    # Factor 7
    "ST268Q07JA", "ST268Q09JA", "ST268Q08JA",
    # Factor 8
    "SC204Q02JA", "SC204Q05JA", "SC204Q01JA", "SC204Q06JA",
    # Factor 9
    "SC155Q09HA", "SC155Q08HA", "SC155Q07HA", "SC155Q06HA", "SC155Q10HA", "SC155Q11HA",
    # Factor 11
    "ST263Q06JA", "ST263Q04JA", "ST263Q08JA", "ST263Q02JA",
    # Factores no escojidos
    "Factor_3_Score", "Factor_4_Score", "Factor_5_Score", "Factor_6_Score", "Factor_10_Score", "Factor_12_Score", "Factor_13_Score",
    "Factor_14_Score", "Factor_15_Score", "Factor_16_Score", "Factor_17_Score", "Factor_18_Score", "Factor_19_Score", "Factor_20_Score",
    "Factor_21_Score", "Factor_22_Score", "Factor_23_Score", "Factor_24_Score", "Factor_25_Score", "Factor_26_Score"
]

df_with_factors_modified = discart_var(df_with_factors, discart_list)

# GENERAR ARCHIVO PARA ANALISIS DE CLASES LATENTES
df_with_factors_modified.to_csv("df_with_factors.csv", index=False)

* Exportar Variables *

In [26]:
df_with_factors_modified

,CNTSCHID,CNTSTUID,STRATUM,BOOKID,RDESIGN,RCORE_TEST,RCORE_PERF,RS1_LEV,RS1_TEST,RCO1S_PERF,...,PV8SCIC,PV9SCIC,PV10SCIC,VER_DAT,Factor_1_Score,Factor_2_Score,Factor_7_Score,Factor_8_Score,Factor_9_Score,Factor_11_Score
0,60400034.0,60400002.0,PER01,Form 5,Design A (Stage1 then Stage2),Testlet 1,Low Level,Low,Testlet 14,Low Level,...,398.469,405.563,364.602,02MAY23:13:48:32,-0.493444,0.783784,1.205973,0.633266,0.685374,2.369148
1,60400267.0,60400006.0,PER02,Form 15,Not used,Not used,Not used,Not used,Not used,Not used,...,385.746,338.499,358.010,02MAY23:13:48:33,0.094325,1.407910,-0.868510,0.571156,-0.291186,-0.498986
2,60400224.0,60400008.0,PER01,Form 43,Not used,Not used,Not used,Not used,Not used,Not used,...,384.335,304.553,315.926,02MAY23:13:48:32,-0.236149,-0.807549,1.083039,-0.995118,0.065036,0.214088
3,60400187.0,60400010.0,PER02,Form 4,Design A (Stage1 then Stage2),Testlet 1,Medium Level,Low,Testlet 14,Low Level,...,339.249,315.577,372.093,02MAY23:13:48:33,0.077280,-0.970616,0.893987,0.034888,-1.028894,0.833396
4,60400102.0,60400011.0,PER03,Form 24,Not used,Not used,Not used,Not used,Not used,Not used,...,411.513,431.318,400.626,02MAY23:13:48:33,-0.335316,0.660466,0.765561,-0.471327,0.978359,0.108292
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6963,60400176.0,60409527.0,PER01,Form 44,Not used,Not used,Not used,Not used,Not used,Not used,...,327.241,393.202,463.055,02MAY23:13:48:32,0.252040,-2.300492,0.874722,0.374019,-0.788221,-1.209264
6964,60400010.0,60409529.0,PER02,Form 9,Design A (Stage1 then Stage2),Testlet 3,Medium Level,High,Testlet 13,Low Level,...,436.983,434.046,362.291,02MAY23:13:48:33,-0.337162,-0.436094,-0.971631,-0.992547,-0.509457,-0.912271
6965,60400277.0,60409531.0,PER03,Form 8,Design B (Stage2 then Stage1),Testlet 5,Medium Level,High,Testlet 12,Medium Level,...,487.241,410.435,493.333,02MAY23:13:48:34,-0.532792,-0.324412,0.558886,0.931641,1.317473,-0.852753
6966,60400177.0,60409533.0,PER02,Form 22,Not used,Not used,Not used,Not used,Not used,Not used,...,443.615,421.701,420.300,02MAY23:13:48:33,1.682772,2.070901,-1.648625,-4.378855,-1.612695,-0.715372


In [28]:
# PASO 6.5: df_afe
# ===========================================

# GENERAR ARCHIVO PARA ANALISIS DE CLASES LATENTES
df_afe.to_csv("df_afe.csv", index=False)